# 01 — Split par Patient & Baseline 2D (Coupe par Coupe, Binaire)

**Objectif :** Construire une baseline propre, reproductible et sans *data leakage* pour la détection binaire de la démence (`0 = Non Demented` vs `1 = Demented`) sur le dataset IRM **OASIS-1**, en entraînant un CNN 2D coupe par coupe sur le GPU **Intel Iris (`/XPU:0`)**.

### Plan du notebook
1. **Configuration & GPU Intel XPU** (`tf-iris-fedora`).
2. **Split strict par patient (`subject_id`)** : Train (70 %) / Val (15 %) / Test (15 %), sauvegardé dans `splits_binary.csv` pour être réutilisé à l'identique dans le notebook 2 (*modèle 61 coupes*).
3. **Pipeline `tf.data`** optimisé pour 7,4 Go de RAM partagée (streaming depuis le disque, 1 canal niveaux de gris, résolution `96 × 192` préservant le ratio 2:1, data augmentation IRM).
4. **Modèle CNN 2D Baseline** avec une couche `slice_embedding` (128 dims) réutilisable pour le modèle 61 coupes.
5. **Entraînement** avec `class_weight` pour le déséquilibre et suivi de la ROC-AUC / PR-AUC.
6. **Évaluation complète sur le jeu de Test** (patients jamais vus) à 3 niveaux : **coupe**, **volume MPR** (moyenne de 61 coupes) et **patient**, avec détail du rappel par sévérité (`Very mild`, `Mild`, `Moderate`).


## 1. Configuration & Vérification du GPU Intel XPU


In [ ]:
import os
import re
import time
import warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

# Reproductibilité
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paramètres globaux
DATA_DIR = Path("/home/timba/Téléchargements/Data")
OUT_DIR = DATA_DIR / "artifacts_modeling"
OUT_DIR.mkdir(exist_ok=True)

IMG_H, IMG_W = 96, 192  # Ratio exact 2:1 (identique à 248x496), léger en RAM/VRAM
BATCH_SIZE = 32
EPOCHS = 12
STEPS_PER_EPOCH = 120   # ~3840 coupes variées par époque
RETRAIN = True          # True pour entraîner la version régularisée (anti-overfitting)

plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

devices = tf.config.list_physical_devices()
print("TensorFlow :", tf.__version__)
print("Périphériques :", devices)
DEVICE = "/XPU:0" if tf.config.list_physical_devices("XPU") else ("/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0")
print(f"✅ Périphérique sélectionné : {DEVICE}")


## 2. Indexation & Split Strict par Patient (`subject_id`)

- Cible binaire : `target = 0` (`Non Demented`) vs `target = 1` (`Very mild`, `Mild`, `Moderate Dementia`).
- Le split est effectué **au niveau des 347 patients uniques**, en stratifiant par sous-classe (avec `Mild + Moderate` regroupées pour la stratification car `Moderate` n'a que 2 patients), afin de garantir **0 patient partagé** entre Train, Val et Test.
- Le fichier `artifacts_modeling/splits_binary.csv` est sauvegardé pour garantir une comparaison 100 % équitable avec le modèle 61 coupes.


In [ ]:
CLASS_ORDER = ["Non Demented", "Very mild Dementia", "Mild Dementia", "Moderate Dementia"]
PAT = re.compile(r"^(OAS1_\d+)_(MR\d+)_(mpr-\d+)_(\d+)\.(\w+)$")

rows = []
for cls in CLASS_ORDER:
    for p in (DATA_DIR / cls).iterdir():
        m = PAT.match(p.name)
        if m:
            subj, sess, mpr, slc, _ = m.groups()
            rows.append({
                "filepath": str(p),
                "label_4cls": cls,
                "target": 0 if cls == "Non Demented" else 1,
                "subject_id": subj,
                "session": sess,
                "mpr_id": mpr,
                "volume_id": f"{subj}_{sess}_{mpr}",
                "slice_idx": int(slc),
            })

df = pd.DataFrame(rows).sort_values(["volume_id", "slice_idx"]).reset_index(drop=True)

# Table au niveau patient
patients = (
    df.groupby("subject_id", as_index=False)
    .agg(label_4cls=("label_4cls", "first"), target=("target", "first"), n_images=("filepath", "count"))
)
# Clé de stratification (Mild + Moderate regroupées pour le tirage stratifié)
patients["strat_key"] = patients["label_4cls"].replace({"Moderate Dementia": "Mild+Mod", "Mild Dementia": "Mild+Mod"})

# 70% Train / 30% Temp, puis 15% Val / 15% Test
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(sss1.split(patients, patients["strat_key"]))

temp_patients = patients.iloc[temp_idx].reset_index(drop=True)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(sss2.split(temp_patients, temp_patients["strat_key"]))

train_subjs = set(patients.iloc[train_idx]["subject_id"])
val_subjs = set(temp_patients.iloc[val_rel]["subject_id"])
test_subjs = set(temp_patients.iloc[test_rel]["subject_id"])

# Vérification stricte de l'absence de fuite
assert len(train_subjs & val_subjs) == 0
assert len(train_subjs & test_subjs) == 0
assert len(val_subjs & test_subjs) == 0
assert len(train_subjs | val_subjs | test_subjs) == len(patients)

df["split"] = df["subject_id"].map(
    lambda s: "train" if s in train_subjs else ("val" if s in val_subjs else "test")
)

split_csv = OUT_DIR / "splits_binary.csv"
df.to_csv(split_csv, index=False)
print(f"✅ Split sauvegardé dans : {split_csv}")

# Résumé par split
summary_split = (
    df.groupby(["split", "target"])
    .agg(n_patients=("subject_id", "nunique"), n_volumes=("volume_id", "nunique"), n_images=("filepath", "count"))
    .reset_index()
)
display(summary_split)

print("\nRépartition des sous-classes (en nombre de patients) par split :")
display(
    df.drop_duplicates("subject_id")
    .pivot_table(index="label_4cls", columns="split", values="subject_id", aggfunc="count", fill_value=0)[["train", "val", "test"]]
)


## 3. Pipeline `tf.data` & Gestion du Déséquilibre

- Les images sont chargées **à la volée** depuis le disque en **1 canal** (`channels=1`, puisque R = G = B), redimensionnées en `96 × 192` et normalisées dans `[0, 1]`.
- Sur le jeu d'entraînement, on applique une **data augmentation modérée** adaptée aux IRM cérébrales (translation légère, zoom léger, petite variation de contraste/luminosité — pas de flip vertical).
- Pour accélérer la validation à chaque époque, on évalue pendant `fit()` sur **1 coupe sur 3** (21 coupes/volume), puis on évalue à la fin sur **100 % des 61 coupes** de Val et Test.


In [ ]:
# Mélange complet de train_df AVANT tf.data (évite le biais de fenêtre sur les volumes triés)
train_df = df[df["split"] == "train"].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

# Sous-ensemble rapide pour le suivi de validation pendant l'entraînement (1 coupe sur 3)
val_fast_df = val_df[val_df["slice_idx"] % 3 == 0].reset_index(drop=True)

# Poids de classe pour compenser le déséquilibre (~77% sains vs ~23% malades)
n_neg = (train_df["target"] == 0).sum()
n_pos = (train_df["target"] == 1).sum()
total = len(train_df)
class_weight = {
    0: total / (2.0 * n_neg),
    1: total / (2.0 * n_pos),
}
print(f"Train : {len(train_df):,} images | Val (fast) : {len(val_fast_df):,} | Val (full) : {len(val_df):,} | Test (full) : {len(test_df):,}")
print(f"Class weights : {{0 (Non Demented): {class_weight[0]:.3f}, 1 (Demented): {class_weight[1]:.3f}}}")

def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=1)
    img = tf.image.resize(img, [IMG_H, IMG_W], method="bilinear")
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.cast(label, tf.float32)

def augment_image(img, label):
    # Augmentation renforcée anti-overfitting (exécutée sur CPU dans tf.data -> compatible ITEX XPU)
    # 1. Flip horizontal (symétrie gauche-droite du cerveau -> double la diversité anatomique)
    img = tf.image.random_flip_left_right(img)
    # 2. Translation aléatoire par pad + random_crop (jusqu'à ±6 px)
    img = tf.image.resize_with_crop_or_pad(img, IMG_H + 12, IMG_W + 16)
    img = tf.image.random_crop(img, size=[IMG_H, IMG_W, 1])
    # 3. Variations modérées de luminosité / contraste + léger bruit gaussien
    img = tf.image.random_brightness(img, max_delta=0.10)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = img + tf.random.normal(tf.shape(img), mean=0.0, stddev=0.02)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(sub_df, training=False, batch_size=BATCH_SIZE):
    paths = sub_df["filepath"].values
    labels = sub_df["target"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(sub_df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.repeat()
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_fast_ds = make_dataset(val_fast_df, training=False)
val_full_ds = make_dataset(val_df, training=False, batch_size=64)
test_full_ds = make_dataset(test_df, training=False, batch_size=64)

# Vérification d'un batch
xb, yb = next(iter(train_ds))
print(f"Batch shape : X={xb.shape}, y={yb.shape}, min={xb.numpy().min():.3f}, max={xb.numpy().max():.3f}")


## 4. Architecture CNN 2D Baseline

Un CNN 2D compact à 4 blocs convolutionnels (`32 → 64 → 128 → 192`) avec `BatchNormalization`, `MaxPooling2D`, `SpatialDropout2D`, et une couche **`slice_embedding` (128 unités)** juste avant la sortie sigmoïde.

Cette couche `slice_embedding` sera directement réutilisée dans le notebook 2 pour extraire la séquence `(61, 128)` de chaque volume MPR.


In [ ]:
def build_baseline_cnn(input_shape=(IMG_H, IMG_W, 1)):
    reg = keras.regularizers.l2(1e-4)
    inp = keras.Input(shape=input_shape, name="mri_slice")
    x = inp

    for i, (filters, drop) in enumerate([(32, 0.10), (64, 0.15), (128, 0.20), (160, 0.25)], start=1):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=reg, name=f"conv{i}_a")(x)
        x = layers.BatchNormalization(name=f"bn{i}_a")(x)
        x = layers.ReLU(name=f"relu{i}_a")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=reg, name=f"conv{i}_b")(x)
        x = layers.BatchNormalization(name=f"bn{i}_b")(x)
        x = layers.ReLU(name=f"relu{i}_b")(x)
        x = layers.MaxPooling2D(2, name=f"pool{i}")(x)
        x = layers.SpatialDropout2D(drop, name=f"sdrop{i}")(x)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.35, name="drop_gap")(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=reg, name="slice_embedding")(x)
    x = layers.Dropout(0.50, name="drop_head")(x)
    out = layers.Dense(1, activation="sigmoid", name="prob_demented")(x)
    return keras.Model(inp, out, name="Baseline_2D_CNN")

with tf.device(DEVICE):
    model = build_baseline_cnn()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=8e-4),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[
            keras.metrics.AUC(name="roc_auc", curve="ROC"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )

model.summary()


## 5. Entraînement sur GPU Intel XPU


In [ ]:
model_path = OUT_DIR / "baseline_2d_cnn.keras"

if model_path.exists() and not RETRAIN:
    print(f"♻️ Modèle déjà entraîné trouvé : {model_path} (RETRAIN=False).")
    print("   Passer RETRAIN = True en section 1 pour relancer un entraînement complet.")
else:
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_roc_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_roc_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_roc_auc",
            mode="max",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    t0 = time.time()
    with tf.device(DEVICE):
        history = model.fit(
            train_ds,
            steps_per_epoch=STEPS_PER_EPOCH,
            epochs=EPOCHS,
            validation_data=val_fast_ds,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=2,
        )
    elapsed = time.time() - t0
    print(f"\n✅ Entraînement terminé en {elapsed/60:.1f} min. Meilleur modèle sauvegardé : {model_path}")

    # Courbes d'apprentissage
    hist = history.history
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, title in zip(
        axes,
        ["loss", "roc_auc", "pr_auc"],
        ["Binary Cross-Entropy", "ROC-AUC (niveau coupe)", "PR-AUC (niveau coupe)"],
    ):
        ax.plot(hist[metric], label="Train", lw=2)
        ax.plot(hist[f"val_{metric}"], label="Val (fast)", lw=2)
        ax.set_title(title)
        ax.set_xlabel("Époque")
        ax.legend()
    plt.tight_layout()
    plt.show()


## 6. Évaluation Complète : Coupe → Volume MPR → Patient

On calibre le seuil optimal (maximisant le F1 / Youden-J) sur le jeu de **Validation**, puis on évalue sans biais sur le jeu de **Test** (patients totalement inédits) aux 3 niveaux :
1. **Coupe (2D)**
2. **Volume MPR** (moyenne des 61 coupes)
3. **Patient** (moyenne des probabilités de tous les volumes du patient)


In [ ]:
# Prédictions complètes sur Val et Test (100% des coupes)
best_model = keras.models.load_model(model_path)

with tf.device(DEVICE):
    val_df["pred_prob"] = best_model.predict(val_full_ds, verbose=0).ravel()
    test_df["pred_prob"] = best_model.predict(test_full_ds, verbose=0).ravel()

# Sauvegarde des prédictions pour comparaison ultérieure
val_df[["filepath", "subject_id", "volume_id", "slice_idx", "label_4cls", "target", "pred_prob"]].to_csv(
    OUT_DIR / "baseline_2d_val_preds.csv", index=False
)
test_df[["filepath", "subject_id", "volume_id", "slice_idx", "label_4cls", "target", "pred_prob"]].to_csv(
    OUT_DIR / "baseline_2d_test_preds.csv", index=False
)

# Seuil optimal calibré sur Validation au niveau patient (indice de Youden : TPR - FPR)
val_pat = val_df.groupby("subject_id", as_index=False).agg(target=("target", "first"), pred_prob=("pred_prob", "mean"))
fpr_v, tpr_v, thr_v = roc_curve(val_pat["target"], val_pat["pred_prob"])
best_thr = float(thr_v[np.argmax(tpr_v - fpr_v)])
print(f"Seuil optimal calibré sur Validation (niveau patient) : {best_thr:.3f}")

def eval_level(y_true, y_prob, thr, name):
    y_pred = (y_prob >= thr).astype(int)
    return {
        "Niveau": name,
        "N": len(y_true),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "F1 (Malade)": f1_score(y_true, y_pred, zero_division=0),
    }

# Agrégation Volume et Patient sur le Test set
test_vol = test_df.groupby("volume_id", as_index=False).agg(
    subject_id=("subject_id", "first"), label_4cls=("label_4cls", "first"),
    target=("target", "first"), pred_prob=("pred_prob", "mean")
)
test_pat = test_df.groupby("subject_id", as_index=False).agg(
    label_4cls=("label_4cls", "first"), target=("target", "first"), pred_prob=("pred_prob", "mean")
)

results = pd.DataFrame([
    eval_level(test_df["target"], test_df["pred_prob"], best_thr, "Test — Coupe (2D)"),
    eval_level(test_vol["target"], test_vol["pred_prob"], best_thr, "Test — Volume MPR (moy. 61 coupes)"),
    eval_level(test_pat["target"], test_pat["pred_prob"], best_thr, "Test — Patient (agrégation finale)"),
])
display(results.round(4))

print("\nRapport détaillé sur le TEST SET (niveau Patient) :")
y_pat_pred = (test_pat["pred_prob"] >= best_thr).astype(int)
print(classification_report(test_pat["target"], y_pat_pred, target_names=["Non Demented (0)", "Demented (1)"], digits=3))

# Rappel par sous-classe clinique sur le Test set (niveau Patient)
test_pat["pred"] = y_pat_pred
print("Taux de détection par sous-classe clinique (Test — niveau Patient) :")
display(
    test_pat.groupby("label_4cls")
    .agg(n_patients=("subject_id", "count"), proba_moyenne=("pred_prob", "mean"), taux_detecte_malade=("pred", "mean"))
    .round(3)
)


In [ ]:
# Visualisation : Courbe ROC + Matrice de confusion + Distribution des probabilités (Test — niveau Patient)
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))

# 1. Courbe ROC (Coupe vs Volume vs Patient)
for name, y_t, y_p, ls in [
    ("Coupe (2D)", test_df["target"], test_df["pred_prob"], ":"),
    ("Volume (61 coupes)", test_vol["target"], test_vol["pred_prob"], "--"),
    ("Patient", test_pat["target"], test_pat["pred_prob"], "-"),
]:
    fpr, tpr, _ = roc_curve(y_t, y_p)
    auc = roc_auc_score(y_t, y_p)
    axes[0].plot(fpr, tpr, linestyle=ls, lw=2, label=f"{name} (AUC={auc:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("Courbes ROC — Test Set")
axes[0].set_xlabel("Taux de Faux Positifs")
axes[0].set_ylabel("Taux de Vrais Positifs")
axes[0].legend(fontsize=8.5)

# 2. Matrice de confusion (niveau Patient)
cm = confusion_matrix(test_pat["target"], y_pat_pred)
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["Prédit Sain", "Prédit Malade"])
axes[1].set_yticklabels(["Vrai Sain", "Vrai Malade"])
axes[1].set_title(f"Matrice de confusion — Patients (seuil={best_thr:.2f})")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14, fontweight="bold",
                     color="white" if cm[i, j] > cm.max()/2 else "black")

# 3. Distribution des probabilités prédites par vraie classe (Patients Test)
for cls_val, name, color in [(0, "Non Demented", "#2a9d8f"), (1, "Demented", "#e76f51")]:
    vals = test_pat.loc[test_pat["target"] == cls_val, "pred_prob"]
    axes[2].hist(vals, bins=12, range=(0, 1), alpha=0.65, label=name, color=color, edgecolor="black")
axes[2].axvline(best_thr, color="black", linestyle="--", lw=1.5, label=f"Seuil = {best_thr:.2f}")
axes[2].set_title("Probas moyennes prédites — Patients Test")
axes[2].set_xlabel("P(Demented)")
axes[2].set_ylabel("Nombre de patients")
axes[2].legend(fontsize=8.5)

plt.tight_layout()
plt.show()
